# 03 — Splits

Creates the five splits used by all downstream training:

1. **Random** measurement-level 80/10/10.
2. **T-extrapolation**: per eligible (solute, solvent) pair (≥5 meas AND ΔT ≥ 20 K), the upper 25 % of the pair's T range is held out for test; lower 75 % goes to train + val.
3. **Cold-solute** 80/10/10 at the solute level.
4. **Cold-pair** 80/10/10 at the (solute, solvent) pair level.
5. **Cold-solvent** 80/10/10 at the solvent level.

Every split ends with an explicit leakage audit.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

RES  = Path.cwd().parent / 'results'
DATA = Path.cwd().parent / 'data' / 'BigSolDBv2.0.csv'

data = np.load(RES / 'features.npz', allow_pickle=True)
T       = data['T']
pair_id = data['pair_id']
n = len(T)
print(f'loaded {n:,} rows')

## Split 1 — random 80/10/10

In [ ]:
rng = np.random.default_rng(42)
perm = rng.permutation(n)
n_train = int(round(0.80 * n))
n_val   = int(round(0.10 * n))
random_train = np.sort(perm[:n_train])
random_val   = np.sort(perm[n_train : n_train + n_val])
random_test  = np.sort(perm[n_train + n_val :])
print(f'random  train={len(random_train):,}  val={len(random_val):,}  test={len(random_test):,}')

## Split 2 — T-extrapolation

Rule per eligible pair (≥5 measurements AND ΔT ≥ 20 K):
- `T_bound = T_min + 0.75 · (T_max − T_min)`
- Rows with `T > T_bound` → test
- Rows with `T ≤ T_bound` → holdable → randomly split 90/10 into train/val

Ineligible pairs (single T, ΔT<20 K, or <5 measurements) go entirely to train.

In [ ]:
sort_idx = np.argsort(pair_id, kind='stable')
sorted_pair = pair_id[sort_idx]
sorted_T = T[sort_idx]
starts = np.concatenate([[0], np.where(np.diff(sorted_pair) != 0)[0] + 1, [len(sorted_pair)]])
pair_ids_unique = sorted_pair[starts[:-1]]

pair_meta = {}
for k, p in enumerate(pair_ids_unique):
    a, b = starts[k], starts[k + 1]
    Tp = sorted_T[a:b]
    pair_meta[int(p)] = (b - a, float(Tp.min()), float(Tp.max()))

MIN_MEAS       = 5
MIN_DT         = 20.0
UPPER_FRACTION = 0.25

eligible_pairs = {
    p for p, (nm, tmin, tmax) in pair_meta.items()
    if nm >= MIN_MEAS and (tmax - tmin) >= MIN_DT
}
print(f'eligible pairs: {len(eligible_pairs):,} / {len(pair_meta):,}')

In [ ]:
is_test     = np.zeros(n, dtype=bool)
is_holdable = np.zeros(n, dtype=bool)

for p in eligible_pairs:
    nm, tmin, tmax = pair_meta[p]
    T_bound = tmin + (1.0 - UPPER_FRACTION) * (tmax - tmin)
    rows = np.where(pair_id == p)[0]
    Tp = T[rows]
    upper = rows[Tp > T_bound]
    lower = rows[Tp <= T_bound]
    if len(upper) == 0 or len(lower) == 0:
        continue
    is_test[upper] = True
    is_holdable[lower] = True

holdable_idx = np.where(is_holdable)[0]
rng_val = np.random.default_rng(43)
pick = rng_val.permutation(len(holdable_idx))
n_val2 = int(round(0.10 * len(holdable_idx)))
val_rows = holdable_idx[pick[:n_val2]]
train_holdable = holdable_idx[pick[n_val2:]]

other_train = np.where(~is_test & ~is_holdable)[0]

textrap_train = np.sort(np.concatenate([train_holdable, other_train]))
textrap_val   = np.sort(val_rows)
textrap_test  = np.sort(np.where(is_test)[0])
print(f'textrap train={len(textrap_train):,}  val={len(textrap_val):,}  test={len(textrap_test):,}')

if len(textrap_test) > 0:
    T_test_ = T[textrap_test]
    T_train_hold = T[np.intersect1d(textrap_train, np.where(is_holdable)[0])]
    print(f'  eligible-pair rows only: train mean T = {T_train_hold.mean():.2f} K, '
          f'test mean T = {T_test_.mean():.2f} K')

## Cold-group splits

For each of solute / pair / solvent: shuffle groups, take 80% train / 10% val / 10% test.

In [ ]:
df_full = pd.read_csv(DATA)
df_full = df_full.dropna(subset=['LogS(mol/L)']).reset_index(drop=True)
assert len(df_full) == n

solute_id,  solute_uniques  = pd.factorize(df_full['SMILES_Solute'].astype(str))
solvent_id, solvent_uniques = pd.factorize(df_full['SMILES_Solvent'].astype(str))

n_solutes = len(solute_uniques)
n_solvents = len(solvent_uniques)
n_pairs_total = len(pair_meta)
print(f'solutes={n_solutes:,}  solvents={n_solvents:,}  pairs={n_pairs_total:,}')

In [ ]:
def group_split(group_ids, n_groups, seed, frac_train=0.80, frac_val=0.10):
    rng_local = np.random.default_rng(seed)
    order = rng_local.permutation(n_groups)
    n_tr = int(round(frac_train * n_groups))
    n_va = int(round(frac_val   * n_groups))
    tr_g = set(order[:n_tr].tolist())
    va_g = set(order[n_tr : n_tr + n_va].tolist())
    te_g = set(order[n_tr + n_va :].tolist())
    tr = np.array([g in tr_g for g in group_ids])
    va = np.array([g in va_g for g in group_ids])
    te = np.array([g in te_g for g in group_ids])
    return np.where(tr)[0], np.where(va)[0], np.where(te)[0], (len(tr_g), len(va_g), len(te_g))

coldsol_train,  coldsol_val,  coldsol_test,  csg = group_split(solute_id,  n_solutes,  seed=101)
coldpair_train, coldpair_val, coldpair_test, cpg = group_split(pair_id,    n_pairs_total, seed=102)
coldsolv_train, coldsolv_val, coldsolv_test, cvg = group_split(solvent_id, n_solvents, seed=103)

print(f'cold-solute  groups train/val/test = {csg}')
print(f'cold-pair    groups train/val/test = {cpg}')
print(f'cold-solvent groups train/val/test = {cvg}')

## Leakage audit

Every disjointness contract is asserted here — if any is violated the cell will raise.

In [ ]:
def audit(name, tr, va, te, expect_disjoint):
    print(f'--- {name} ---')
    all_rows = np.concatenate([tr, va, te])
    assert len(np.unique(all_rows)) == len(all_rows), f'{name}: index overlap'
    assert len(all_rows) == n, f'{name}: rows lost'
    print(f'  rows: train={len(tr):,}  val={len(va):,}  test={len(te):,}  total={len(all_rows):,}')
    for label, arr in (('train', tr), ('val', va), ('test', te)):
        print(f'  {label}: solutes={len(set(solute_id[arr].tolist()))}  '
              f'solvents={len(set(solvent_id[arr].tolist()))}  '
              f'pairs={len(set(pair_id[arr].tolist()))}')
    for other in ('val', 'test'):
        other_arr = va if other == 'val' else te
        sol_o  = len(set(solute_id[tr].tolist())  & set(solute_id[other_arr].tolist()))
        solv_o = len(set(solvent_id[tr].tolist()) & set(solvent_id[other_arr].tolist()))
        pair_o = len(set(pair_id[tr].tolist())    & set(pair_id[other_arr].tolist()))
        print(f'  train ∩ {other}: solutes={sol_o}  solvents={solv_o}  pairs={pair_o}')
    for other in ('val', 'test'):
        other_arr = va if other == 'val' else te
        for kind, ids in (('solutes', solute_id), ('solvents', solvent_id), ('pairs', pair_id)):
            if kind not in expect_disjoint:
                continue
            overlap = set(ids[tr].tolist()) & set(ids[other_arr].tolist())
            assert not overlap, f'{name}: LEAKAGE — {kind}(train) ∩ {kind}({other}) has {len(overlap)}'
    print(f'  ✓ {sorted(expect_disjoint)} disjoint\n')

audit('random',       random_train,   random_val,   random_test,   set())
audit('textrap',      textrap_train,  textrap_val,  textrap_test,  set())
audit('cold-solute',  coldsol_train,  coldsol_val,  coldsol_test,  {'solutes','pairs'})
audit('cold-pair',    coldpair_train, coldpair_val, coldpair_test, {'pairs'})
audit('cold-solvent', coldsolv_train, coldsolv_val, coldsolv_test, {'solvents','pairs'})

## Save

In [ ]:
np.savez_compressed(
    RES / 'splits.npz',
    random_train=random_train, random_val=random_val, random_test=random_test,
    textrap_train=textrap_train, textrap_val=textrap_val, textrap_test=textrap_test,
    coldsol_train=coldsol_train, coldsol_val=coldsol_val, coldsol_test=coldsol_test,
    coldpair_train=coldpair_train, coldpair_val=coldpair_val, coldpair_test=coldpair_test,
    coldsolv_train=coldsolv_train, coldsolv_val=coldsolv_val, coldsolv_test=coldsolv_test,
)
print('saved splits.npz')